# Zadanie 3: optymalizacja dyskretna

Termin realizacji: 20 kwietnia 2026

Zadanie do oddania przez MS Teams. Do oddania: kod oraz krótkie sprawozdanie w PDF (można na przykład przy użyciu `quarto render notebook.ipynb --to pdf`).

## Na 3.0

Do realizacji:

1. Zaimplementuj dyskretny problem plecakowy z trzema plecakami w MiniZinc na podstawie przykładu (plik `minizinc.ipynb`). Spróbuj rozwiązać problem dla 10 zestawów parametrów o różnych wielkościach tak, aby rozwiązanie największego problemu trwało powyżej 5 sekund. Zanotuj w każdym przypadku liczbę wszystkich przedmiotów, pojemności plecaków, liczbę wybranych przedmiotów i sumaryczną wartość przedmiotów w każdym plecaku osobno.
2. Losowanie przedmiotów w każdym przypadku powinno być tak ustawione, aby optymalne rozwiązanie wymagało wzięcia przynajmniej dwóch przedmiotów do każdego plecaka oraz zostawienia przynajmniej dwóch przedmiotów poza plecakami. W raporcie należy umieścić informację w jaki sposób zostało to zweryfikowane.
3. Zmodyfikuj metodę z notatnika `tabu_search.ipynb` tak aby rozwiązywała opisywany problem plecakowy. Porównaj na tych samych problemach czy Minizinc i Tabu search zwracają równie dobre rozwiazania, oraz wypisz jakie to są rozwiązania. Wykonaj eksperymenty z trzema różnymi długościami listy zakazów (1, 2, 5).

## Na 4.0

Do realizacji:

1. Punkty z zadania na 3.0.
2. Rozszerz możliwe ruchy w tabu search o przeniesienie przedmiotu z jednego plecaka do drugiego. Zapisz rozważ czy to poprawia działanie metody (czy znalezione jest lepsze, takie samo czy gorsze rozwiązanie? czy rozwiązanie jest znajdowane szybciej czy wolniej?). Dla każdego z 10 zestawów parametrów problemu plecakowego wykonaj ocenę przez uśrednienie dla 10 różnych losowych przypadków.
3. Podsumuj dane w formie tabelki z czterema kolumnami (Minizinc, tabu search z listą o długości 1, 2, i 5) oraz 10 wierszami (po jednym dla zestawu parametrów problemu), a w komórkach umieść średnią wartość wartości przedmiotów oraz średni czas potrzebny do uzyskania rozwiązania.

## Na 5.0

Do realizacji:

1. Punkty z zadania na 4.0.
2. Zaimplementuj samodzielnie algorytm symulowanego wyżarzania z podobnym interfejsem co tabu search. Porównaj jego działanie do rozważanych wcześniej rozwiązań dla trzech różnych schematów chłodzenia. Dobierz liczbę iteracji tak, aby czas działania był porównywalny do tabu search.


# Na 3.0

## Generacja 10 losowych przypadków
### Parametry:
* 3 plecaki
* 10 zestawów parametrów
* największy - conajmniej 10 sek.
* optymalne rozwiązanie: 
    * kady plecak ma pojemośc 2 razy lub więcej większą od wagi kazdego przedmiotu
    * suma pojemnosci plecakow < suma wag - 2 najwieksze wagi

In [41]:
# import Pkg; Pkg.add("Distributions")
# import Pkg; Pkg.add("DataStructures")

In [ ]:
using Distributions

#     gościu sugeruje, że losować 3 kategorie przedmiotów - małe waggi i małe wartości, średnie wagi, i duże wagi i że naturalnie te constrainty się spełnią
#     do modelu w minizincu dodać funckje sprawdzające czy roziwązanie ma spełnione te zasady i dodać je jako zmienne do rozwiązania
#     eksperymenty
#     minizinc api do julii - istnieje!!

function make_dzn(n::Int, n_knapsacks::Int)
    profits = rand(DiscreteUniform(10, 1000), n)
    weights = rand(DiscreteUniform(10, 100), n)
    capacities = rand(DiscreteUniform(100, 300), n_knapsacks)
    content = """
    ITEM = _(1..$n);
    knapsack_n = $n_knapsacks;
    capacities = $capacities;
    profits = $profits;
    weights = $weights;
    """
    file = open("knapsack_generated.dzn", "w+")
    write(file, content)
    close(file)
end
make_dzn(100, 3)

In [43]:
struct KnapsackProblem
    capacities::Vector{Int}
    weights::Vector{Int}
    profits::Vector{Int}
end
# Zrobić zamiast tego wczytywanie z pliku .dzn
function generate_problem(n_knapsacks = 3, n_items = 100)
    capacities = rand(DiscreteUniform(2500, 3000), n_knapsacks)
    profits = rand(DiscreteUniform(10, 1000), n_items)
    weights = rand(DiscreteUniform(10, 100), n_items)
    kp = KnapsackProblem(capacities, profits, weights)
end

kp1 = generate_problem()

KnapsackProblem([2899, 2634, 2521], [529, 409, 483, 219, 958, 583, 968, 110, 591, 616  …  602, 992, 188, 58, 54, 335, 349, 484, 616, 354], [62, 26, 41, 43, 19, 47, 17, 37, 86, 43  …  22, 30, 29, 73, 50, 94, 31, 61, 100, 59])

In [ ]:
using Random
using Printf

# Pomocniczy parser: liczby całkowite z zapisu [1, 2, 3].
function _parse_int_vector(txt::AbstractString)
    m = collect(eachmatch(r"-?\d+", txt))
    return [parse(Int, mm.match) for mm in m]
end

# Wczytanie wartości skalarnej z pliku DZN, np. knapsack_n = 3;
function _parse_dzn_scalar_int(content::String, key::String)
    rx = Regex(key * raw"\s*=\s*(-?\d+)\s*;")
    m = match(rx, content)
    m === nothing && error("Brak pola $(key) w pliku DZN")
    return parse(Int, m.captures[1])
end

# Wczytanie tablicy z pliku DZN, np. capacities = [10, 20, 30];
function _parse_dzn_vector_int(content::String, key::String)
    rx = Regex(key * raw"\s*=\s*\[(.*?)\]\s*;", "s")
    m = match(rx, content)
    m === nothing && error("Brak tablicy $(key) w pliku DZN")
    return _parse_int_vector(m.captures[1])
end

function load_knapsack_from_dzn(path::String)
    content = read(path, String)
    capacities = _parse_dzn_vector_int(content, "capacities")
    profits = _parse_dzn_vector_int(content, "profits")
    weights = _parse_dzn_vector_int(content, "weights")
    return KnapsackProblem(capacities, weights, profits)
end

function make_dzn_three_categories(path::String, n_items::Int, n_knapsacks::Int=3; rng=Random.default_rng())
    n_items >= 8 || error("Dla warunku \"co najmniej 2 poza plecakami\" potrzebujemy >= 8 przedmiotów")

    # Trzy kategorie: małe, średnie, duże.
    n_small = max(2, round(Int, 0.35 * n_items))
    n_medium = max(2, round(Int, 0.35 * n_items))
    n_large = n_items - n_small - n_medium
    n_large < 2 && (n_large = 2; n_medium = n_items - n_small - n_large)

    small_weights = rand(rng, 3:12, n_small)
    medium_weights = rand(rng, 13:35, n_medium)
    large_weights = rand(rng, 36:90, n_large)

    small_profits = rand(rng, 8:40, n_small)
    medium_profits = rand(rng, 45:170, n_medium)
    large_profits = rand(rng, 180:620, n_large)

    weights = vcat(small_weights, medium_weights, large_weights)
    profits = vcat(small_profits, medium_profits, large_profits)

    perm = randperm(rng, n_items)
    weights = weights[perm]
    profits = profits[perm]

    total_weight = sum(weights)
    max_weight = maximum(weights)

    # Pojemności dobieramy tak, aby:
    # 1) każdy plecak mógł pomieścić min. 2 przedmioty,
    # 2) łączna pojemność była wyraźnie mniejsza niż suma wag (zostają elementy poza plecakami).
    min_cap = 2 * max_weight + 5
    center = max(min_cap, round(Int, 0.56 * total_weight / n_knapsacks))
    capacities = [max(min_cap, center + rand(rng, -15:15)) for _ in 1:n_knapsacks]

    content = """
    ITEM = _(1..$n_items);
    knapsack_n = $n_knapsacks;
    capacities = $(repr(capacities));
    profits = $(repr(profits));
    weights = $(repr(weights));
    """

    open(path, "w") do io
        write(io, content)
    end

    return (capacities=capacities, weights=weights, profits=profits)
end

In [ ]:
function verify_assignment_rules(assignment::Vector{Int}, n_knapsacks::Int)
    counts = [count(==(k), assignment) for k in 1:n_knapsacks]
    outside = count(==(0), assignment)
    ok = all(c -> c >= 2, counts) && outside >= 2
    return (ok=ok, items_in_knapsack=counts, items_outside=outside)
end

function generate_report_problem_set(problem_specs::Vector{Tuple{String,Int}};
    problems_dir::String="problems",
    model_path::String="knapsack_report.mzn",
    max_attempts_per_case::Int=60,
    largest_min_seconds::Float64=10.0,
    solver::String="coinbc")

    mkpath(problems_dir)
    accepted = String[]

    for (idx, (name, n_items)) in enumerate(problem_specs)
        target_seconds = idx == length(problem_specs) ? largest_min_seconds : 0.0
        accepted_case = false
        local current_items = n_items

        for attempt in 1:max_attempts_per_case
            dzn_path = joinpath(problems_dir, "$(name).dzn")
            make_dzn_three_categories(dzn_path, current_items, 3)

            mz = run_minizinc_from_julia(model_path, dzn_path; solver=solver)

            if mz.profit === nothing || isempty(mz.assignment)
                if idx == length(problem_specs) && attempt % 10 == 0
                    current_items = round(Int, 1.15 * current_items)
                end
                continue
            end

            local_check = verify_assignment_rules(mz.assignment, 3)
            ok = mz.packing_rules_ok && local_check.ok && (mz.elapsed_seconds >= target_seconds)

            if ok
                push!(accepted, dzn_path)
                println(@sprintf("[OK] %s | items=%d | t=%.3fs | in=%s | out=%d | profit=%d",
                    name, current_items, mz.elapsed_seconds, string(mz.items_in_knapsack), mz.items_outside, mz.profit))
                accepted_case = true
                break
            end

            if idx == length(problem_specs) && attempt % 10 == 0
                current_items = round(Int, 1.15 * current_items)
            end
        end

        if !accepted_case
            println("[WARN] Nie udało się zaakceptować przypadku $(name) w limicie prób")
        end
    end

    return accepted
end

In [ ]:
problem_specs = [
    ("kp_01", 35),
    ("kp_02", 45),
    ("kp_03", 55),
    ("kp_04", 70),
    ("kp_05", 85),
    ("kp_06", 100),
    ("kp_07", 120),
    ("kp_08", 145),
    ("kp_09", 170),
    ("kp_10", 210)
]

generated_files = generate_report_problem_set(problem_specs;
    problems_dir="problems",
    model_path="knapsack_report.mzn",
    max_attempts_per_case=80,
    largest_min_seconds=10.0,
    solver="coinbc"
)

println("\nZaakceptowane problemy:")
foreach(println, generated_files)

## TabuSearch

In [44]:
using DataStructures
using Distributions

mutable struct TabuState{TMove,P,TF}
    tabu_buffer::CircularBuffer{TMove}
    best_seen::P
    best_seen_obj::TF
    current::P
    considered::P
    iter::Int
end

function TabuState(p, x0; buffer_length::Int=10)
    moves = possible_moves(p, x0)
    obj = objective(p, x0)
    return TabuState{eltype(moves),typeof(x0),typeof(obj)}(
        CircularBuffer{eltype(moves)}(buffer_length), x0, obj, copy(x0), copy(x0), 1
    )
end


function solve_tabu(p, s::TabuState; iteration_limit::Int=100)
    iter_best_move = 0
    while s.iter < iteration_limit
        moves = possible_moves(p, s.current)
        best_move = 0
        best_move_obj = Inf
        for (i_move, move) in enumerate(moves)
            if in(move, s.tabu_buffer)
                # move forbidden, do not consider
                continue
            end
            # evaluate move
            copyto!(s.considered, s.current)
            apply!(s.considered, move)
            considered_value = objective(p, s.considered)
            if considered_value < best_move_obj
                best_move = i_move
                best_move_obj = considered_value
            end
        end
        # no allowed move found
        if best_move == 0
            break
        end
        chosen_move = moves[best_move]
        item_id = chosen_move[1]
        prev_bin = s.current[item_id]
        apply!(s.current, chosen_move)
        push!(s.tabu_buffer, invert_move(p, moves[best_move], prev_bin))
        if best_move_obj < s.best_seen_obj
            # best so far, let's remember it
            iter_best_move = s.iter
            copyto!(s.best_seen, s.current)
            s.best_seen_obj = best_move_obj
        end
        s.iter += 1
    end
    println("Iteracja w której zostało osiągnięte najlepsze rozwiązanie: ", iter_best_move)
    return s.best_seen
end


function objective(p::KnapsackProblem, x::Vector{Int})
    profit = 0
    for i in eachindex(x)
        if x[i] > 0
            profit += p.profits[i]
        end
    end
    return -profit
end


function apply!(x, move::Tuple{Int,Int})
    item_idx, new_bin = move
    x[item_idx] = new_bin
    return x
end

function invert_move(::KnapsackProblem, move::Tuple{Int,Int}, old_bin::Int)
    return (move[1], old_bin)
end


function possible_moves(p::KnapsackProblem, x::Vector{Int})
    move_list = Tuple{Int,Int}[]
    current_weights = zeros(Int, length(p.capacities)) #sum(p.weights .* x)
    # add item
    for (item_id, bin_id) in enumerate(x)
        if bin_id > 0 # 0 oznacza brak przypisania
            current_weights[bin_id] += p.weights[item_id]
        end
    end    

    for i in eachindex(x)
        if x[i] > 0
            # możliwość usunięcia i z plecaka x[i]
            push!(move_list, (i, 0))
        end
        for k in eachindex(p.capacities)
            # przeniesienie do plecaka
            if x[i] == 0 && (current_weights[k] + p.weights[i] <= p.capacities[k])
                push!(move_list, (i,k))
            end
        end
    end

    return move_list
end


possible_moves (generic function with 1 method)

In [45]:
function test(kp)
    x0 = fill(0, length(kp.weights))
    st = TabuState(kp, x0; buffer_length=10)
    sol = solve_tabu(kp, st; iteration_limit=1000000)

    println("Pełne rozwiązanie: ", sol)
    
    println("Best objective: ", st.best_seen_obj)
    println("Last iteration: ", st.iter)
end
@time begin
test(kp1)
end



Iteracja w której zostało osiągnięte najlepsze rozwiązanie: 196
Pełne rozwiązanie: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 3, 2, 0, 0, 2, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 3, 1, 0, 2, 0, 0, 0, 0, 3, 1, 1, 3, 0, 2, 0, 1, 0, 2, 0, 0, 0, 0, 0, 3, 2, 1, 0, 0, 1, 0]
Best objective: -1882
Last iteration: 1000000
  2.670311 seconds (6.22 M allocations: 1018.125 MiB, 2.94% gc time, 2.92% compilation time)


### 3.3 - rózne długości linii zakazów

In [46]:
function experiment(kp)
    dlugosci = [1, 2, 5]
    
    for L in dlugosci
        @time begin
        println("\n--- Eksperyment dla długości tabu L = $L ---")
        
        x0 = fill(0, length(kp.weights))
        # Tutaj przekazujemy L do parametru buffer_length
        st = TabuState(kp, x0; buffer_length = L)
        
        sol = solve_tabu(kp, st; iteration_limit = 100000)
        
        println("Pełne rozwiązanie: ", sol)
    
        println("Best objective: ", st.best_seen_obj)
        println("Last iteration: ", st.iter)
        end
    end
end
experiment(kp1)


--- Eksperyment dla długości tabu L = 1 ---
Iteracja w której zostało osiągnięte najlepsze rozwiązanie: 21
Pełne rozwiązanie: [0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 1, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 2, 1, 0, 2, 0, 0, 0, 0, 2, 1, 1, 3, 0, 1, 0, 1, 0, 2, 0, 0, 0, 0, 0, 2, 3, 1, 0, 0, 1, 0]
Best objective: -1735
Last iteration: 100000
  0.218906 seconds (601.47 k allocations: 90.241 MiB, 3.94% gc time)

--- Eksperyment dla długości tabu L = 2 ---
Iteracja w której zostało osiągnięte najlepsze rozwiązanie: 21
Pełne rozwiązanie: [0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 1, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 2, 1, 0, 2, 0, 0, 0, 0, 2, 1, 1, 3, 0, 1, 0, 1, 0, 2, 0, 0, 0, 0, 0, 2, 3, 1, 0, 0, 1, 0]
Best objective: -1735
Last ite

In [ ]:
function solve_tabu_lengths(kp::KnapsackProblem; tabu_lengths::Vector{Int}=[1, 2, 5], iteration_limit::Int=100000)
    out = Dict{Int, Any}()
    for L in tabu_lengths
        x0 = fill(0, length(kp.weights))
        st = TabuState(kp, x0; buffer_length=L)
        t0 = time()
        sol = solve_tabu(kp, st; iteration_limit=iteration_limit)
        elapsed = time() - t0
        out[L] = (
            assignment = sol,
            profit = -st.best_seen_obj,
            elapsed_seconds = elapsed
        )
    end
    return out
end

## MiniZinc

In [ ]:
function run_minizinc_from_julia(model_path::String, dzn_path::String; solver::String="coinbc", time_limit_ms::Int=0)
    cmd_with_solver = time_limit_ms > 0 ?
        `minizinc --solver $solver --time-limit $time_limit_ms $model_path $dzn_path` :
        `minizinc --solver $solver $model_path $dzn_path`
    cmd_default = time_limit_ms > 0 ?
        `minizinc --time-limit $time_limit_ms $model_path $dzn_path` :
        `minizinc $model_path $dzn_path`

    t0 = time()
    raw = ""
    try
        raw = read(cmd_with_solver, String)
    catch e
        msg = sprint(showerror, e)
        if occursin("no solver with tag", lowercase(msg))
            raw = read(cmd_default, String)
        else
            rethrow(e)
        end
    end
    elapsed = time() - t0

    function get_line_value(key::String)
        rx = Regex("^" * key * "=(.*)\$", "m")
        m = match(rx, raw)
        m === nothing && return nothing
        return strip(m.captures[1])
    end

    profit_txt = get_line_value("profit")
    x_txt = get_line_value("x")
    in_txt = get_line_value("items_in_knapsack")
    out_txt = get_line_value("items_outside")
    ok_txt = get_line_value("packing_rules_ok")

    return (
        raw_output = raw,
        elapsed_seconds = elapsed,
        profit = profit_txt === nothing ? nothing : parse(Int, profit_txt),
        assignment = x_txt === nothing ? Int[] : _parse_int_vector(x_txt),
        items_in_knapsack = in_txt === nothing ? Int[] : _parse_int_vector(in_txt),
        items_outside = out_txt === nothing ? nothing : parse(Int, out_txt),
        packing_rules_ok = ok_txt === nothing ? false : lowercase(ok_txt) == "true"
    )
end

compare_methods_on_files (generic function with 1 method)

## Porównanie metod - eksperymenty:

In [ ]:
function compare_methods_on_files(files::Vector{String};
    model_path::String="knapsack_report.mzn",
    solver::String="coinbc",
    tabu_lengths::Vector{Int}=[1, 2, 5],
    iteration_limit::Int=100000)

    rows = NamedTuple[]

    for path in files
        kp = load_knapsack_from_dzn(path)

        mz = run_minizinc_from_julia(model_path, path; solver=solver)
        tabu = solve_tabu_lengths(kp; tabu_lengths=tabu_lengths, iteration_limit=iteration_limit)

        println("\n=== $(basename(path)) ===")
        println(@sprintf("MiniZinc: profit=%s, time=%.3fs, in=%s, out=%s, rules=%s",
            string(mz.profit), mz.elapsed_seconds, string(mz.items_in_knapsack), string(mz.items_outside), string(mz.packing_rules_ok)))

        for L in tabu_lengths
            t = tabu[L]
            println(@sprintf("Tabu L=%d: profit=%d, time=%.3fs", L, t.profit, t.elapsed_seconds))
            push!(rows, (
                file = basename(path),
                method = "Tabu_L$(L)",
                profit = t.profit,
                elapsed_seconds = t.elapsed_seconds
            ))
        end

        push!(rows, (
            file = basename(path),
            method = "MiniZinc",
            profit = mz.profit === nothing ? -1 : mz.profit,
            elapsed_seconds = mz.elapsed_seconds
        ))
    end

    return rows
end

In [49]:
if !@isdefined(generated_files)
    generated_files = sort(filter(f -> endswith(f, ".dzn"), readdir("problems"; join=true)))
end

comparison_rows = compare_methods_on_files(generated_files;
    model_path="knapsack_report.mzn",
    solver="coinbc",
    tabu_lengths=[1, 2, 5],
    iteration_limit=100000
)

println("\nLiczba rekordów porównania: ", length(comparison_rows))
first_n = min(length(comparison_rows), 8)
println("Przykładowe rekordy:")
for i in 1:first_n
    println(comparison_rows[i])
end

Iteracja w której zostało osiągnięte najlepsze rozwiązanie: 14
Iteracja w której zostało osiągnięte najlepsze rozwiązanie: 14
Iteracja w której zostało osiągnięte najlepsze rozwiązanie: 18

=== kp_01.dzn ===
MiniZinc: profit=4430, time=1.125s, in=[4, 6, 4], out=21, rules=true
Tabu L=1: profit=4314, time=0.091s
Tabu L=2: profit=4314, time=0.088s
Tabu L=5: profit=4356, time=0.110s
Iteracja w której zostało osiągnięte najlepsze rozwiązanie: 15
Iteracja w której zostało osiągnięte najlepsze rozwiązanie: 15
Iteracja w której zostało osiągnięte najlepsze rozwiązanie: 15

=== kp_02.dzn ===
MiniZinc: profit=5790, time=0.947s, in=[6, 5, 4], out=30, rules=true
Tabu L=1: profit=5784, time=0.130s
Tabu L=2: profit=5784, time=0.132s
Tabu L=5: profit=5784, time=0.139s
Iteracja w której zostało osiągnięte najlepsze rozwiązanie: 21
Iteracja w której zostało osiągnięte najlepsze rozwiązanie: 25
Iteracja w której zostało osiągnięte najlepsze rozwiązanie: 25

=== kp_03.dzn ===
MiniZinc: profit=6506, time=

# Na 4.0

In [50]:
function possible_moves(p::KnapsackProblem, x::Vector{Int})
    move_list = Tuple{Int,Int}[]
    current_weights = zeros(Int, length(p.capacities)) #sum(p.weights .* x)
    # add item
    for (item_id, bin_id) in enumerate(x)
        if bin_id > 0 # 0 oznacza brak przypisania
            current_weights[bin_id] += p.weights[item_id]
        end
    end    

    for i in eachindex(x)
        if x[i] > 0
            # możliwość usunięcia i z plecaka x[i]
            push!(move_list, (i, 0))
        end
        for k in eachindex(p.capacities)
            # przeniesienie do plecaka
            if x[i] == 0 && (current_weights[k] + p.weights[i] <= p.capacities[k])
                push!(move_list, (i,k))
            end
            # dodatkowa możliwość - przeniesienie do innego plecaka, do punktu 4.0
            if x[i] != k && (current_weights[k] + p.weights[i] <= p.capacities[k])
                push!(move_list, (i, k))
            end
        end
    end

    return move_list
end

possible_moves (generic function with 1 method)

## Na 5.0

## Symulowane wyzazanie - TabuSearch

In [51]:
mutable struct SAState{P, TF}
    current::P
    best_seen::P
    best_seen_obj::TF
    temp::Float64
    iter::Int
end
# Konstruktor pomocniczy
function SAState(p, x0; initial_temp::Float64=100.0)
    obj = objective(p, x0)
    return SAState{typeof(x0), typeof(obj)}(
        copy(x0),
        copy(x0),
        obj,
        initial_temp,
        1
    )
end
function solve_simulated_annealing(p, s::SAState; 
                                   max_iter=10000, 
                                   cooling_rate=0.995)
    iter_best_move = 0
    while s.iter < max_iter

        moves = possible_moves(p, s.current)
        if isempty(moves) break end
        move = rand(moves) 
        old_obj = objective(p, s.current)
        
        test_state = copy(s.current)
        apply!(test_state, move)
        new_obj = objective(p, test_state)
        
        ΔE = new_obj - old_obj
        
        if ΔE < 0 || rand() < exp(-ΔE / s.temp)
            s.current = test_state
            
            if new_obj < s.best_seen_obj
                iter_best_move = s.iter
                s.best_seen = copy(s.current)
                s.best_seen_obj = new_obj
            end
        end
        
        s.temp *= cooling_rate
        s.iter += 1
    end
    println("Iteracja w której zostało osiągnięte najlepsze rozwiązanie: ", iter_best_move)
    return s.best_seen
end

solve_simulated_annealing (generic function with 1 method)

In [52]:

function test2(kp)
    x0 = fill(0, length(kp.weights))
    st = SAState(kp, x0)
    sol = solve_simulated_annealing(kp, st; max_iter=1000000)

    println("Full solution: ", sol)
    
    println("Best objective: ", st.best_seen_obj)
    println("Last iteration: ", st.iter)
end
@time begin
test2(kp1)
end

Iteracja w której zostało osiągnięte najlepsze rozwiązanie: 489
Full solution: [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 2, 1, 3, 0, 0, 1, 1, 3, 2, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 2, 0, 1, 3, 3, 1, 0, 0, 0, 2, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 2, 3, 0, 0, 0, 0, 0, 2, 2, 3, 0, 0, 1, 0, 2, 0, 1, 0, 0, 0, 0, 0, 2, 3, 2, 0, 0, 0, 1]
Best objective: -1979
Last iteration: 1000000
  1.298053 seconds (8.10 M allocations: 1.749 GiB, 8.41% gc time, 4.68% compilation time)
